# IOAI — 2026 Mock Grid Classification (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-mock-grid-classification/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Grid Classification — 모범답안 (에지 기반 격자선 탐지, 비지도)

궁격도는 여러 서브이미지를 격자로 붙여 **내부에 강한 직선 경계(가로/세로 분할선)**가 생긴다. 각 열/행의
평균 에지 강도 프로파일을 구해, 테두리를 제외한 내부에 **뾰족한 피크**(균일한 분할선)가 있으면 격자로 판정한다.
라벨 없이도 동작(비지도). 손라벨 held-out accuracy ≈ 0.85 (전부 0 인 0.5 대비).

In [ ]:
import pandas as pd, numpy as np, os
from PIL import Image
train = pd.read_csv("data/train.csv")            # filename (라벨 없음)
test_list = pd.read_csv("data/test_list.csv")["filename"].tolist()
print("train", len(train), "| test", len(test_list))

In [ ]:
def is_grid(path, thresh=4.3):
    im = np.asarray(Image.open(os.path.join("data/test", path)).convert("L").resize((256,256)), float)
    gx = np.abs(np.diff(im, axis=1)).mean(0)      # 열별 평균 에지(세로선 후보)
    gy = np.abs(np.diff(im, axis=0)).mean(1)      # 행별 평균 에지(가로선 후보)
    peak = lambda pr: pr[15:-15].max() / (np.median(pr) + 1e-6)   # 내부 최대피크 / 중앙값
    return 1 if max(peak(gx), peak(gy)) > thresh else 0

pred = [is_grid(f) for f in test_list]
pd.DataFrame({"filename": test_list, "label": pred}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred), "| grid predicted", sum(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)